# Chapter 15 Lab — Softmax and Cross-Entropy

Can arbitrary class scores become one probability distribution, and how should a model be penalized for confident mistakes? Read [`blog.md`](<blog.md>) first.

## Prediction

Before running code, predict the largest probability for `[2, 5, 1]`, whether adding 100 changes softmax, and which loss is larger: `-log(0.8)` or `-log(0.2)`.

In [ ]:
import numpy as np

logits = np.array([2.0, 5.0, 1.0])
shifted = logits - logits.max()
probs = np.exp(shifted) / np.exp(shifted).sum()
print(probs)
assert np.isclose(probs.sum(), 1.0)
assert np.argmax(probs) == 1

## Mathematics

Softmax: $p_i=e^{z_i}/\sum_j e^{z_j}$. Stable form: subtract the maximum logit first. Cross-entropy: $L=-\sum_i y_i\log p_i$.

In [ ]:
# Manual calculation for logits [2, 5, 1]
e = np.exp(np.array([2.0, 5.0, 1.0]))
manual = e / e.sum()
assert np.allclose(manual, np.array([0.047314, 0.946499, 0.006188]), atol=1e-5)

target = np.array([0.0, 1.0, 0.0])
loss = -np.sum(target * np.log(manual))
assert np.isclose(loss, -np.log(manual[1]))

In [ ]:
def softmax_from_scratch(x):
    m = max(x)
    exp_x = [np.exp(v - m) for v in x]
    total = sum(exp_x)
    return np.array([v / total for v in exp_x])

scratch = softmax_from_scratch([2.0, 5.0, 1.0])
assert np.allclose(scratch, manual)

In [ ]:
import matplotlib.pyplot as plt

labels = ['Apartment', 'Villa', 'Farmhouse']
plt.bar(labels, manual)
plt.ylabel('Probability')
plt.title('Softmax for logits [2, 5, 1]')
plt.show()

In [ ]:
def stable_softmax(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - x.max())
    return e / e.sum()

base = stable_softmax(np.array([2.0, 5.0, 1.0]))
shifted = stable_softmax(np.array([102.0, 105.0, 101.0]))
assert np.allclose(base, shifted)

In [ ]:
# Change exactly one thing: multiply all logits by 2
colder = stable_softmax(np.array([2.0, 5.0, 1.0]) * 2)
print('original:', base)
print('scaled logits:', colder)

## Observe

The largest logit receives the largest probability. Adding the same constant changes nothing. Multiplying logits by 2 makes the distribution more concentrated.

## Explain

Softmax depends only on logit differences. Cross-entropy reads the true-class probability, so reducing that probability increases the loss sharply.

In [ ]:
# Level 4–5 challenge
# YOUR CODE HERE
# Implement temperature softmax: exp(logits / T) / sum(exp(logits / T))
# Then verify that T > 1 produces a softer distribution than T = 1.

## Reflection

- [ ] I can distinguish logits from probabilities.
- [ ] I can implement numerically stable softmax from scratch.
- [ ] I can compute cross-entropy for a one-hot target.
- [ ] I understand why the combined gradient is prediction minus truth.

Next: remove the global model rule and let nearby examples vote.